In [ ]:
# Instalações (Google Colab)
!apt-get update > /dev/null
!apt install chromium-chromedriver -y > /dev/null
!cp /usr/lib/chromium-browser/chromedriver /usr/bin
!pip install selenium pandas openpyxl > /dev/null

# Imports
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import os, re, requests, zipfile
import pandas as pd
from datetime import datetime

# Setup de pastas
data_hoje = datetime.now().strftime('%Y-%m-%d')
pasta_base = f"{data_hoje}_joli"
pasta_imagens = os.path.join(pasta_base, "imagens")
os.makedirs(pasta_imagens, exist_ok=True)

# Configurar WebDriver
options = Options()
options.binary_location = "/usr/bin/chromium-browser"
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 10)

# COLETAR LINKS (sem limite)
categoria_url = "https://www.joli.com.br/pisos-revestimentos-e-porcelanatos?page="
links = []
pagina = 1

print("🔎 Coletando todos os links dos produtos da Joli...")

while True:
    driver.get(categoria_url + str(pagina))
    try:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".vtex-search-result-3-x-galleryItem a")))
    except:
        print(f"⚠️ Elementos não encontrados na página {pagina}, encerrando a coleta.")
        break

    produtos = driver.find_elements(By.CSS_SELECTOR, ".vtex-search-result-3-x-galleryItem a")

    novos_links = []
    for p in produtos:
        try:
            href = p.get_attribute("href")
            if href and href not in links:
                novos_links.append(href)
        except:
            continue

    if not novos_links:
        print(f"✅ Fim da coleta na página {pagina}")
        break

    links.extend(novos_links)
    print(f"🔗 Página {pagina}: {len(novos_links)} novos links")

    pagina += 1

# Função para extrair detalhes com Selenium
def extrair_detalhes_com_selenium(driver, url):
    try:
        driver.get(url)
        try:
            wait.until(EC.presence_of_element_located((By.TAG_NAME, "h1")))
        except:
            print(f"⚠️ Produto não carregado: {url}")
            return None

        nome = driver.find_element(By.TAG_NAME, "h1").text.strip()

        try:
            codigo = driver.find_element(By.CSS_SELECTOR, "span.vtex-product-identifier-0-x-product-identifier__value").text.strip()
        except:
            codigo = "0"

        try:
            preco_original = driver.find_element(By.CSS_SELECTOR, ".joli-joli-theme-1-x-list-price .joli-joli-theme-1-x-price").text.strip()
        except:
            preco_original = "N/A"

        try:
            preco_promocional = driver.find_element(By.CSS_SELECTOR, ".joli-joli-theme-1-x-selling-price .joli-joli-theme-1-x-price").text.strip()
        except:
            preco_promocional = "N/A"

        preco_m2 = preco_promocional

        try:
            descricao = driver.find_element(By.CLASS_NAME, "joli-joli-theme-1-x-description").text.strip()
        except:
            try:
                descricao = driver.find_element(By.CLASS_NAME, "joli-joli-theme-1-x-product-description").text.strip()
            except:
                descricao = "Descrição não encontrada"

        especificacoes = []
        try:
            blocos = driver.find_elements(By.CLASS_NAME, "joli-joli-theme-1-x-specification")
            for bloco in blocos:
                nome_esp = bloco.find_element(By.CLASS_NAME, "joli-joli-theme-1-x-specificationName").text.strip()
                valor = bloco.find_element(By.CLASS_NAME, "joli-joli-theme-1-x-specificationValue").text.strip()
                especificacoes.append(f"{nome_esp}: {valor}")
            especificacoes_str = " | ".join(especificacoes)
        except:
            especificacoes_str = "N/A"

        try:
            img_url = driver.find_element(By.CSS_SELECTOR, "img.joli-store-components-0-x-productImageTag--main").get_attribute("src")
        except:
            img_url = "N/A"

        return [nome, codigo, especificacoes_str, descricao, preco_original, preco_promocional, preco_m2, img_url, url]

    except Exception as e:
        print(f"❌ Erro em {url}: {e}")
        return None

# Função para salvar imagem
def salvar_imagem(url_img, nome_base):
    if url_img == "N/A": return
    try:
        resposta = requests.get(url_img, timeout=10)
        if resposta.status_code == 200:
            caminho = os.path.join(pasta_imagens, f"{nome_base}.jpg")
            with open(caminho, 'wb') as f:
                f.write(resposta.content)
    except Exception as e:
        print(f"❌ Erro ao baixar imagem: {e}")

# COLETAR DADOS
dados_produtos = []
for i, link in enumerate(links):
    print(f"[{i+1}/{len(links)}] Coletando dados de: {link}")
    resultado = extrair_detalhes_com_selenium(driver, link)
    if resultado:
        dados_produtos.append(resultado)
        codigo = re.sub(r'[\\/*?:"<>|]', "", resultado[1])
        if not codigo or codigo == "0":
            codigo = re.sub(r'[\\/*?:"<>|]', "", resultado[0])[:50]
        salvar_imagem(resultado[7], codigo)

driver.quit()

# Criar DataFrame
df = pd.DataFrame(dados_produtos, columns=[
    "Nome", "Código", "Especificações", "Descrição", "Preço Original",
    "Preço Promocional", "Preço por m²", "Imagem", "URL"
])

# Limpar preços
def limpar_preco(valor):
    try:
        return float(valor.replace("R$", "").replace(".", "").replace(",", ".").strip())
    except:
        return None

df["Preço Original"] = df["Preço Original"].apply(limpar_preco)
df["Preço Promocional"] = df["Preço Promocional"].apply(limpar_preco)
df["Preço por m²"] = df["Preço por m²"].apply(limpar_preco)

# Adicionar coluna da loja
df["Loja"] = "Joli"

# Reorganizar colunas
df = df[[
    "Código", "Nome", "Preço Promocional", "Preço Original", "Preço por m²",
    "Especificações", "Descrição", "Imagem", "URL", "Loja"
]]

# Salvar CSV com separador ';'
csv_path = os.path.join(pasta_base, f"joli_{data_hoje}.csv")
df.to_csv(csv_path, sep=';', index=False, encoding='utf-8-sig')
print(f"\n📄 CSV salvo em: {csv_path}")

# Salvar também como Excel
xlsx_path = os.path.join(pasta_base, f"joli_{data_hoje}.xlsx")
df.to_excel(xlsx_path, index=False)
print(f"📊 Excel salvo em: {xlsx_path}")

# Compactar imagens
zip_path = os.path.join(pasta_base, f"imagens_joli_{data_hoje}.zip")
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for root, dirs, files in os.walk(pasta_imagens):
        for file in files:
            zipf.write(os.path.join(root, file), arcname=file)

print(f"📦 Imagens compactadas em: {zip_path}")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


cp: '/usr/lib/chromium-browser/chromedriver' and '/usr/bin/chromedriver' are the same file
🔎 Coletando todos os links dos produtos da Joli...
🔗 Página 1: 6 novos links
🔗 Página 2: 6 novos links
🔗 Página 3: 6 novos links
🔗 Página 4: 6 novos links
🔗 Página 5: 6 novos links
🔗 Página 6: 6 novos links
🔗 Página 7: 6 novos links
🔗 Página 8: 6 novos links
